# Speaker Feedback


## Setup

Loads environment variables and adds `src/` to the Python path so the package imports cleanly.


In [ ]:
from pathlib import Path
import sys
import json
import os

try:
    from dotenv import load_dotenv
    load_dotenv(override=True)
except Exception as exc:
    print("dotenv not available:", exc)

def find_project_root(start: Path) -> Path:
    for parent in [start, *start.parents]:
        if (parent / "src" / "speaker_feedback").exists():
            return parent

    env_root = os.environ.get("SPEAKER_FEEDBACK_ROOT") or os.environ.get("PROJECT_ROOT")
    if env_root:
        candidate = Path(env_root)
        if (candidate / "src" / "speaker_feedback").exists():
            return candidate

    for candidate in (Path("/notebooks"), Path.cwd().parent):
        if (candidate / "src" / "speaker_feedback").exists():
            return candidate

    return start

PROJECT_ROOT = find_project_root(Path.cwd())
if not (PROJECT_ROOT / "src" / "speaker_feedback").exists():
    raise FileNotFoundError("Could not find src/speaker_feedback. Set SPEAKER_FEEDBACK_ROOT or PROJECT_ROOT.")

sys.path.insert(0, str(PROJECT_ROOT / "src"))
PROJECT_ROOT


## Thresholds Config

Loads shared thresholds from `src/speaker_feedback/config/thresholds.yml` so all modalities stay consistent.


In [ ]:
from speaker_feedback.config.thresholds import load_thresholds, DEFAULT_THRESHOLDS_PATH

thresholds = load_thresholds(DEFAULT_THRESHOLDS_PATH)

shared_cfg = thresholds["shared"]
speech_cfg = thresholds["speech"]
slides_cfg = thresholds["slides"]
face_cache_cfg = thresholds["face_cache"]
clothing_cfg = thresholds["clothing"]
gaze_cfg = thresholds["gaze"]
emotion_cfg = thresholds["emotion"]
gesture_cfg = thresholds["gesture"]


## Inputs

Set the input video and core speech settings used across the pipeline.

Core inputs:
- `video_path`: path to the presentation video.
- `language`: language hint for speech recognition.
- `whisper_model_size`: model size tradeoff between speed and accuracy.


In [ ]:
video_path = str(PROJECT_ROOT / "data" / "inputs" / "video" / "your_video.mp4")
language = "en"
whisper_model_size = "small"
intelligibility_segment_len = speech_cfg["intelligibility_segment_len"]


## Preview Helpers

Small helpers to show JSON keys and compact previews without dumping full results.


In [ ]:
def summarize_value(v, max_list=2, max_str=160, max_dict_keys=8):
    if isinstance(v, dict):
        return {k: summarize_value(v[k], max_list, max_str, max_dict_keys) for k in list(v)[:max_dict_keys]}
    if isinstance(v, list):
        if len(v) == 0:
            return []
        items = [summarize_value(x, max_list, max_str, max_dict_keys) for x in v[:max_list]]
        if len(v) > max_list:
            items.append("...")
        return items
    if isinstance(v, str):
        return v[:max_str]
    return v


def keys_and_preview(obj):
    if not isinstance(obj, dict):
        return [], {}
    keys = list(obj.keys())
    preview = {k: summarize_value(obj.get(k)) for k in keys}
    return keys, preview


## Speech Analysis

Transcribes the full audio, extracts segment timings, and computes filler usage, speech rate, noise, and intelligibility.


In [ ]:
from speaker_feedback.agents.tools.speech_analysis_tool import analyze_speech_tool

speech_result = analyze_speech_tool(
    video_path=video_path,
    language=language,
    whisper_model_size=whisper_model_size,
    intelligibility_segment_len=intelligibility_segment_len,
)

speech_keys, speech_preview = keys_and_preview(speech_result)
speech_keys, speech_preview


## Slide Analysis (SSIM + OCR + Description + Layout)

Detects slide transitions with SSIM, crops slides via Detectron, then runs OCR and visual understanding on the best frames per slide.


In [ ]:
detectron_model_path = r"D:\path\to\detectron_model.pth"
detectron_config_path = r"D:\path\to\detectron_config.yaml"
ocr_model_id = "Qwen/Qwen2.5-VL-3B-Instruct"

sample_every_sec = slides_cfg["sample_every_sec"]
ssim_thresh = slides_cfg["ssim_thresh"]
min_segment_sec = slides_cfg["min_segment_sec"]
similarity_threshold = slides_cfg["similarity_threshold"]
min_word_count_for_slide = slides_cfg["min_word_count_for_slide"]
ocr_sample_count = slides_cfg["ocr_sample_count"]
top_k_ocr_frames = slides_cfg["top_k_ocr_frames"]
sharpness_weight = slides_cfg["sharpness_weight"]


### OCR Model Download (optional)

Downloads the OCR model into `data/cache/model_cache` so later runs can use local files.


In [ ]:
try:
    from huggingface_hub import snapshot_download
    cache_dir = PROJECT_ROOT / "data" / "cache" / "model_cache"
    cache_dir.mkdir(parents=True, exist_ok=True)
    snapshot_download(repo_id=ocr_model_id, cache_dir=str(cache_dir), local_files_only=False)
    "model download complete"
except Exception as exc:
    str(exc)


In [ ]:
from speaker_feedback.agents.tools.slide_ocr_tool import slide_extraction_tool

slide_result = slide_extraction_tool(
    video_path=video_path,
    model_path=detectron_model_path,
    config_path=detectron_config_path,
    sample_every_sec=sample_every_sec,
    ssim_thresh=ssim_thresh,
    min_segment_sec=min_segment_sec,
    similarity_threshold=similarity_threshold,
    min_word_count_for_slide=min_word_count_for_slide,
    ocr_sample_count=ocr_sample_count,
    top_k_ocr_frames=top_k_ocr_frames,
    sharpness_weight=sharpness_weight,
    ocr_model_id=ocr_model_id,
)

slide_keys, slide_preview = keys_and_preview(slide_result)
slide_keys, slide_preview


## Face Cache

Samples frames per slide and caches detected faces. This is used by clothing, gaze, and emotion modules.


In [ ]:
from speaker_feedback.agents.tools.face_cache_tools import build_face_cache_tool

per_slide_frames = face_cache_cfg["per_slide_frames"]
face_batch_size = face_cache_cfg["face_batch_size"]

face_cache_out = build_face_cache_tool(
    video_path=video_path,
    segments=slide_result.get("segments", []),
    per_slide_frames=per_slide_frames,
    batch_size=face_batch_size,
)

face_cache_keys = list(face_cache_out.keys())
face_cache_preview = {
    "fps": face_cache_out.get("fps"),
    "slide_frame_mapping": summarize_value(face_cache_out.get("slide_frame_mapping", {})),
    "face_crops_cache": {"count": len(face_cache_out.get("face_crops_cache", {}))},
    "dominant_embedding": {"length": len(face_cache_out.get("dominant_embedding", []))},
    "stats": face_cache_out.get("stats", {}),
}
face_cache_keys, face_cache_preview


## Clothing Analysis

Runs CLIP-based multi-attribute outfit analysis on torso crops derived from face boxes.


In [ ]:
from speaker_feedback.agents.tools.clothing_tool import clothing_analysis_tool
from speaker_feedback.video_analysis.clothing_analysis import ClothesCLIP

clip_model_name = clothing_cfg["clip_model_name"]
clip_local_files_only = clothing_cfg["clip_local_files_only"]

clothing_classifier = ClothesCLIP(
    model_name=clip_model_name,
    local_files_only=clip_local_files_only,
)

clothing_frames_per_slide_max = clothing_cfg["clothing_frames_per_slide_max"]
clothing_min_face_conf = clothing_cfg["clothing_min_face_conf"]

clothing_result = clothing_analysis_tool(
    video_path=video_path,
    slide_frame_mapping=face_cache_out.get("slide_frame_mapping", {}),
    face_crops_cache=face_cache_out.get("face_crops_cache", {}),
    clothing_classifier=clothing_classifier,
    frames_per_slide_max=clothing_frames_per_slide_max,
    min_face_conf=clothing_min_face_conf,
)

clothing_keys, clothing_preview = keys_and_preview(clothing_result)
clothing_keys, clothing_preview


## Gaze Analysis

Estimates head pose with MediaPipe FaceMesh + PnP, then maps yaw/pitch into audience, script, or slide-focused gaze.


In [ ]:
from speaker_feedback.agents.tools.gaze_tool import gaze_analysis_tool
from speaker_feedback.video_analysis.gaze_analysis import MediaPipeGazeDirection, GazeHeuristics

idx_to_slide = {
    int(s.get("slide_id")): {
        "slide_content": s.get("ocr_text", ""),
        "start_time": s.get("start_time"),
        "end_time": s.get("end_time"),
    }
    for s in slide_result.get("segments", [])
}

gaze_heuristics = GazeHeuristics(
    pitch_down_thresh=gaze_cfg["pitch_down_thresh"],
    yaw_side_thresh=gaze_cfg["yaw_side_thresh"],
    pitch_center_thresh=gaze_cfg["pitch_center_thresh"],
    pitch_offset=gaze_cfg.get("pitch_offset", 0.0),
    yaw_center_thresh=gaze_cfg["yaw_center_thresh"],
)

gaze_estimator = MediaPipeGazeDirection(heuristics=gaze_heuristics)

gaze_result = gaze_analysis_tool(
    video_path=video_path,
    slide_frame_mapping=face_cache_out.get("slide_frame_mapping", {}),
    face_crops_cache=face_cache_out.get("face_crops_cache", {}),
    gaze_estimator=gaze_estimator,
    idx_to_slide=idx_to_slide,
    frames_per_slide_max=gaze_cfg["frames_per_slide_max"],
    min_face_conf=gaze_cfg["min_face_conf"],
    min_gaze_conf=gaze_cfg["min_gaze_conf"],
    min_valid_frames_per_slide=gaze_cfg["min_valid_frames_per_slide"],
    min_coverage_ratio=gaze_cfg["min_coverage_ratio"],
    min_overall_valid_ratio=gaze_cfg["min_overall_valid_ratio"],
    expand_scale=gaze_cfg["expand_scale"],
    min_face_size=gaze_cfg["min_face_size"],
)

gaze_estimator.close()

gaze_keys, gaze_preview = keys_and_preview(gaze_result)
gaze_keys, gaze_preview


## Emotion Analysis

Uses EmotiEffLib to classify facial emotions on cached face frames and aggregates per-slide and overall statistics.


In [ ]:
from speaker_feedback.agents.tools.emotion_tool import emotion_analysis_tool
from emotiefflib.facial_analysis import EmotiEffLibRecognizer, get_model_list
import torch

emotion_device = "cuda" if torch.cuda.is_available() else "cpu"
emotion_model_name = get_model_list()[0]
fer = EmotiEffLibRecognizer(engine="onnx", model_name=emotion_model_name, device=emotion_device)

emotion_result = emotion_analysis_tool(
    video_path=video_path,
    slide_frame_mapping=face_cache_out.get("slide_frame_mapping", {}),
    face_crops_cache=face_cache_out.get("face_crops_cache", {}),
    fer=fer,
    idx_to_slide=idx_to_slide,
    frames_per_slide_max=emotion_cfg["frames_per_slide_max"],
    min_face_conf=emotion_cfg["min_face_conf"],
    min_valid_frames_per_slide=emotion_cfg["min_valid_frames_per_slide"],
    min_coverage_ratio=emotion_cfg["min_coverage_ratio"],
    min_overall_valid_ratio=emotion_cfg["min_overall_valid_ratio"],
    expand_scale=emotion_cfg["expand_scale"],
    min_face_size=emotion_cfg["min_face_size"],
    batch_size=emotion_cfg["batch_size"],
)

del fer

emotion_keys, emotion_preview = keys_and_preview(emotion_result)
emotion_keys, emotion_preview


## Gesture Analysis

Detects pose keypoints with YOLOv8-Pose and emits timestamped gesture events.


In [ ]:
from speaker_feedback.video_analysis.gesture_analysis import gesture_analysis_tool
from speaker_feedback.video_analysis.gesture_analysis import Gestures, GestureHeuristics

gesture_heuristics = GestureHeuristics(
    hand_to_face_ratio=gesture_cfg["hand_to_face_ratio"],
    arms_crossed_wrist_ratio=gesture_cfg["arms_crossed_wrist_ratio"],
    arms_crossed_chest_y_ratio=gesture_cfg["arms_crossed_chest_y_ratio"],
    open_palms_ratio=gesture_cfg["open_palms_ratio"],
)

gesture_detector = Gestures(
    model_path=gesture_cfg["model_path"],
    conf=gesture_cfg["conf"],
    kp_conf=gesture_cfg["kp_conf"],
    heuristics=gesture_heuristics,
)

gesture_result = gesture_analysis_tool(
    video_path=video_path,
    slide_frame_mapping=face_cache_out.get("slide_frame_mapping", {}),
    gesture_detector=gesture_detector,
    idx_to_slide=idx_to_slide,
    frames_per_slide_max=gesture_cfg["frames_per_slide_max"],
    resize_w=gesture_cfg["resize_w"],
    max_total_frames=gesture_cfg["max_total_frames"],
    min_event_frames=gesture_cfg["min_event_frames"],
    min_pose_coverage_ratio=gesture_cfg["min_pose_coverage_ratio"],
    top_evidence_frames=gesture_cfg["top_evidence_frames"],
)

gesture_keys, gesture_preview = keys_and_preview(gesture_result)
gesture_keys, gesture_preview


## Build Recommendation Payloads

Creates two payloads: visual coaching (gaze, emotion, gesture, clothing) and slide + speech storytelling (alignment between OCR and speech). Each payload includes compact evidence and derived metrics.


In [ ]:
def overlap_sec(a_start, a_end, b_start, b_end):
    return max(0.0, min(a_end, b_end) - max(a_start, b_start))


def weighted_avg(items, value_key, start_key, end_key, seg_start, seg_end):
    total = 0.0
    weight = 0.0
    for item in items:
        s = float(item.get(start_key, 0.0))
        e = float(item.get(end_key, s))
        o = overlap_sec(s, e, seg_start, seg_end)
        if o > 0:
            total += o * float(item.get(value_key, 0.0))
            weight += o
    return (total / weight) if weight > 0 else None


def text_for_interval(segments, seg_start, seg_end, max_chars=300):
    parts = []
    for seg in segments:
        s = float(seg.get("start", 0.0))
        e = float(seg.get("end", s))
        if overlap_sec(s, e, seg_start, seg_end) > 0:
            parts.append(seg.get("text", "").strip())
    text = " ".join([p for p in parts if p])
    return text[:max_chars]


def count_occurrences(occ_list, seg_start, seg_end):
    count = 0
    for occ in occ_list:
        t = float(occ.get("start", 0.0))
        if seg_start <= t <= seg_end:
            count += 1
    return count


def noise_fraction(noise_intervals, seg_start, seg_end):
    dur = max(0.0, seg_end - seg_start)
    if dur == 0:
        return 0.0
    total = 0.0
    for a, b in noise_intervals:
        total += overlap_sec(a, b, seg_start, seg_end)
    return total / dur


def clip_intervals(intervals, seg_start, seg_end):
    out = []
    for a, b in intervals:
        a = float(a)
        b = float(b)
        if overlap_sec(a, b, seg_start, seg_end) > 0:
            out.append([max(a, seg_start), min(b, seg_end)])
    return out


def token_set(text):
    import re
    toks = re.findall(r"[A-Za-z0-9]+", (text or "").lower())
    return set([t for t in toks if t])


def text_similarity(a, b):
    from difflib import SequenceMatcher
    ta = token_set(a)
    tb = token_set(b)
    jaccard = (len(ta & tb) / len(ta | tb)) if (ta or tb) else 0.0
    edit = SequenceMatcher(None, (a or "").lower(), (b or "").lower()).ratio()
    return {
        "jaccard": round(float(jaccard), 3),
        "edit_ratio": round(float(edit), 3),
    }


def text_word_count(text):
    return len([t for t in token_set(text)])


def label_counts(frames, key):
    counts = {}
    for f in frames or []:
        label = f.get(key)
        if label:
            counts[label] = counts.get(label, 0) + 1
    return counts


speech_segments = speech_result.get("segments", [])
rate_windows = speech_result.get("speech_rate", {}).get("windows", [])
intel_segments = speech_result.get("intelligibility", {}).get("per_segment", [])
fillers_words = speech_result.get("filler_occurrences", {}).get("words", [])
fillers_phrases = speech_result.get("filler_occurrences", {}).get("phrases", [])
noise_intervals = speech_result.get("background_noise", {}).get("intervals", [])
slow_intervals = speech_result.get("speech_rate", {}).get("slow", [])
fast_intervals = speech_result.get("speech_rate", {}).get("fast", [])
low_intel_intervals = speech_result.get("intelligibility", {}).get("low_confidence_intervals", [])

per_slide = []
for s in slide_result.get("segments", []):
    start = float(s.get("start_time", 0.0))
    end = float(s.get("end_time", start))
    speech_full = text_for_interval(speech_segments, start, end, max_chars=5000)
    similarity = text_similarity(s.get("ocr_text", ""), speech_full)
    speech_word_count = text_word_count(speech_full)
    ocr_word_count = int(s.get("ocr_word_count", 0))
    speech_overlap_sec = 0.0
    for seg in speech_segments:
        seg_s = float(seg.get("start", 0.0))
        seg_e = float(seg.get("end", seg_s))
        speech_overlap_sec += overlap_sec(seg_s, seg_e, start, end)
    speech_coverage_ratio = (speech_overlap_sec / max(1e-6, end - start))
    per_slide.append({
        "slide_id": s.get("slide_id"),
        "start_time": start,
        "end_time": end,
        "duration": float(s.get("duration", max(0.0, end - start))),
        "ocr_text": s.get("ocr_text", ""),
        "description": s.get("description", ""),
        "layout": s.get("layout", ""),
        "image_path": s.get("image_path"),
        "speech": {
            "wpm": weighted_avg(rate_windows, "wpm", "start", "end", start, end),
            "intelligibility": weighted_avg(intel_segments, "score", "start", "end", start, end),
            "filler_word_count": count_occurrences(fillers_words, start, end),
            "filler_phrase_count": count_occurrences(fillers_phrases, start, end),
            "noise_fraction": noise_fraction(noise_intervals, start, end),
            "speech_text": speech_full,
            "speech_word_count": speech_word_count,
            "speech_overlap_sec": round(float(speech_overlap_sec), 3),
            "speech_coverage_ratio": round(float(speech_coverage_ratio), 3),
            "speech_text_preview": speech_full[:300],
            "evidence_intervals": {
                "slow_speech": clip_intervals(slow_intervals, start, end),
                "fast_speech": clip_intervals(fast_intervals, start, end),
                "low_intelligibility": clip_intervals(low_intel_intervals, start, end),
                "background_noise": clip_intervals(noise_intervals, start, end),
            },
            "filler_occurrences": {
                "words": [f for f in fillers_words if start <= float(f.get("start", 0.0)) <= end][:10],
                "phrases": [f for f in fillers_phrases if start <= float(f.get("start", 0.0)) <= end][:10],
            },
        },
        "content_alignment": {
            "similarity": similarity,
            "ocr_word_count": ocr_word_count,
        },
    })

clothing_summary = None
if "clothing_result" in locals():
    clothing_summary = {
        "summary": clothing_result.get("summary"),
        "description": clothing_result.get("description", ""),
        "style": clothing_result.get("style", ""),
        "top": clothing_result.get("top", ""),
        "is_appropriate": clothing_result.get("is_appropriate"),
        "recommendation": clothing_result.get("recommendation"),
        "detected_attributes": clothing_result.get("detected_attributes", []),
        "coverage": clothing_result.get("coverage", {}),
    }

gaze_summary = None
if "gaze_result" in locals():
    gaze_summary = {
        "overall_summary": gaze_result.get("overall_summary", {}),
        "issues": gaze_result.get("issues", {}),
        "meta": gaze_result.get("meta", {}),
    }

emotion_summary = None
if "emotion_result" in locals():
    emotion_summary = {
        "overall_stats": emotion_result.get("overall_stats", {}),
        "issues": emotion_result.get("issues", {}),
        "meta": emotion_result.get("meta", {}),
    }

gesture_summary = None
if "gesture_result" in locals():
    gesture_summary = {
        "overall": gesture_result.get("overall", {}),
        "issues": gesture_result.get("issues", {}),
        "meta": gesture_result.get("meta", {}),
    }

overall = {
    "speech": {
        "filler_words": speech_result.get("filler_words", {}),
        "filler_phrases": speech_result.get("filler_phrases", {}),
        "intelligibility_global": speech_result.get("intelligibility", {}).get("global_score"),
        "noise_fraction": speech_result.get("background_noise", {}).get("fraction"),
        "avg_wpm": weighted_avg(rate_windows, "wpm", "start", "end", 0.0, float("inf")),
    },
    "slides": {
        "count": len(slide_result.get("segments", [])),
        "avg_words": (
            sum([s.get("ocr_word_count", 0) for s in slide_result.get("segments", [])])
            / max(1, len(slide_result.get("segments", [])))
        ),
    },
    "clothing": clothing_summary,
    "gaze": gaze_summary,
    "emotion": emotion_summary,
    "gesture": gesture_summary,
}

sim_jaccards = [s.get("content_alignment", {}).get("similarity", {}).get("jaccard") for s in per_slide]
sim_edits = [s.get("content_alignment", {}).get("similarity", {}).get("edit_ratio") for s in per_slide]
avg_jaccard = sum([v for v in sim_jaccards if isinstance(v, (int, float))]) / max(1, len(sim_jaccards))
avg_edit = sum([v for v in sim_edits if isinstance(v, (int, float))]) / max(1, len(sim_edits))

multimodal_events = []
g_slides = (gesture_result or {}).get("slide_summaries", {}) if "gesture_result" in locals() else {}
gaze_slides = (gaze_result or {}).get("slide_summaries", {}) if "gaze_result" in locals() else {}
emo_slides = (emotion_result or {}).get("slide_summaries", {}) if "emotion_result" in locals() else {}
for sid, gs in g_slides.items():
    events = gs.get("events", [])
    gaze_ev = (gaze_slides.get(str(sid)) or {}).get("evidence_frames", [])
    emo_ev = (emo_slides.get(str(sid)) or {}).get("evidence_frames", [])
    for ev in events:
        st = float(ev.get("start_time", 0.0))
        en = float(ev.get("end_time", st))
        g_in = [g for g in gaze_ev if st <= float(g.get("t_sec", -1)) <= en]
        e_in = [e for e in emo_ev if st <= float(e.get("t_sec", -1)) <= en]
        multimodal_events.append({
            "slide_id": int(sid),
            "event": ev.get("event"),
            "start_time": st,
            "end_time": en,
            "gaze_labels": label_counts(g_in, "gaze"),
            "emotion_labels": label_counts(e_in, "emotion"),
        })

content_report_payload = {
    "overall": {
        "speech": overall["speech"],
        "slides": overall["slides"],
    },
    "slides": per_slide,
    "derived_metrics": {
        "avg_slide_speech_similarity_jaccard": round(float(avg_jaccard), 3),
        "avg_slide_speech_similarity_edit": round(float(avg_edit), 3),
    },
}

visual_report_payload = {
    "overall": {
        "clothing": clothing_summary,
        "gaze": gaze_summary,
        "emotion": emotion_summary,
        "gesture": gesture_summary,
    },
    "events": {
        "gesture_by_slide": g_slides,
        "gaze_by_slide": gaze_slides,
        "emotion_by_slide": emo_slides,
        "multimodal_events": multimodal_events,
    },
}

content_keys, content_preview = keys_and_preview(content_report_payload)
visual_keys, visual_preview = keys_and_preview(visual_report_payload)
content_keys, content_preview, visual_keys, visual_preview


## Save Payload JSONs

Writes the payloads to `data/outputs/` for later reuse or LLM calls.


In [ ]:
output_dir = PROJECT_ROOT / "data" / "outputs"
output_dir.mkdir(parents=True, exist_ok=True)

content_payload_path = output_dir / "content_report_payload.json"
visual_payload_path = output_dir / "visual_report_payload.json"

with content_payload_path.open("w", encoding="utf-8") as f:
    json.dump(content_report_payload, f, ensure_ascii=True, indent=2)

with visual_payload_path.open("w", encoding="utf-8") as f:
    json.dump(visual_report_payload, f, ensure_ascii=True, indent=2)

content_payload_path, visual_payload_path


## NeMo Recommendations

Optional step that sends each payload to NeMo via a ReAct prompt. Keep `run_nemo = False` if you only want to inspect payloads.


In [ ]:
from speaker_feedback.agents.tools.recommendation_tool import run_nemo_react_recommendations
from speaker_feedback.nemo.tools import build_visual_coaching_input, build_storytelling_input
import json
import time

nemo_config_path = str(PROJECT_ROOT / "src" / "speaker_feedback" / "nemo" / "configs" / "recommendations.yml")

visual_user_input = build_visual_coaching_input(
    payload=visual_report_payload,
    top_k_recommendations=6,
)

story_user_input = build_storytelling_input(
    payload=content_report_payload,
    top_k_recommendations=6,
)

def _print_status(label, start_ts, payload_text=None):
    elapsed = time.time() - start_ts
    size_kb = None
    if payload_text is not None:
        size_kb = len(payload_text.encode("utf-8")) / 1024
    if size_kb is None:
        print(f"[{label}] elapsed={elapsed:.1f}s")
    else:
        print(f"[{label}] elapsed={elapsed:.1f}s, payload={size_kb:.1f} KB")

def _run_report(label, user_input, config_path):
    start = time.time()
    print(f"[{label}] request started...")
    _print_status(label, start, user_input)
    out = run_nemo_react_recommendations(
        config_file=config_path,
        user_input=user_input,
        stream_output=True,
    )
    _print_status(label, start)
    print(f"[{label}] request finished.")
    return out

run_nemo = False
if run_nemo:
    visual_report = _run_report("visual", visual_user_input, nemo_config_path)
    storytelling_report = _run_report("storytelling", story_user_input, nemo_config_path)
    visual_report, storytelling_report


## GPU Cleanup

Runs cleanup helpers to release GPU memory after model-heavy stages.


In [ ]:
try:
    import torch
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        "cuda cache cleared"
except Exception as exc:
    str(exc)


In [ ]:
# Generate combined PDF report
from speaker_feedback.reporting.pdf_report import build_pdf_report

if "storytelling_report" not in globals() or "visual_report" not in globals():
    raise RuntimeError("Run the recommendation cell first to populate storytelling_report and visual_report.")

output_dir = PROJECT_ROOT / "data" / "outputs"
output_dir.mkdir(parents=True, exist_ok=True)
pdf_path = output_dir / "presentation_feedback_report.pdf"

pdf_path = build_pdf_report(
    storytelling_report,
    visual_report,
    content_report_payload,
    visual_report_payload,
    pdf_path,
)
print(f"PDF written to: {pdf_path}")
